## 04 - Model training

### 🎯 Model objective

As stated before, the objective of the project is to have a program that takes as input two reactives in SMILES and gives back a list of the best conditions with their predicted yield for the **Suzuki coupling** reaction. The conditions include the **base, ligand, and solvent** used. For this, the program needs a model that takes as input a **vector of descriptors** containing significant information about the two reactives and gives back the wanted results.


### 🏛️ Notebook Structure

To accomplish this objective, multiple different models and strategies will be tested. Different models will be trained and compared in order to select the best one.

First, preliminary tasks will be performed. The necessary libraries and functions will be imported, and functions to evaluate and compare the models will be defined.

Then for each strategy, the section will contain :

- An explanation and the motivations for choosing the strategy.
- The creation of the required models, their training, and eventual adjustments.
- The testing of the strategy, where its accuracy will be tested.

The final strategy and model will be chosen based on **a comparative accuracy assessment**.

## Preliminary tasks

Before training and testing the models in their different strategies, a few preliminary tasks have to be done.

### 📚 Library and functions imports

A few libraries and functions will be necessary throughout the notebook. 

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import wandb
import torch.nn.functional as F
import pytorch_lightning as pl

# Import necessary functions
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from sklearn.model_selection import GridSearchCV
from torch import nn
from torch.utils.data import DataLoader
from pytorch_lightning.loggers import WandbLogger
from torch.utils.data import Dataset

### 💯 Testing function

To evaluate the models, a function that calculates multiple evaluation metrics is defined.

In [21]:
# Testing function for regressors
def test_model(model, x_test, y_test):
    """
    Test the model on the test dataset and return multiple evaluation metrics.
    Parameters:
    model: The trained model to be evaluated.
    x_test: The test features.
    y_test: The true labels for the test dataset.
    Returns:
    A dictionary containing the evaluation metrics: RMSE, MAE, R2 Score, and Explained Variance Score.
    """
    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    explained_variance = explained_variance_score(y_test, y_pred)

    return {
        "Model": model.__class__.__name__,
        "RMSE": rmse,
        "MAE": mae,
        "R2 Score": r2,
        "Explained Variance Score": explained_variance
    }

# Testing function for neural network model
def test_nn_model(model, x_test, y_test):
    model.eval()
    with torch.no_grad():
        x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
        y_pred = model(x_test_tensor).squeeze().numpy()
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    explained_variance = explained_variance_score(y_test, y_pred)
    
    return {
        "Model": "NeuralNetwork",
        "RMSE": rmse,
        "MAE": mae,
        "R2 Score": r2,
        "Explained Variance Score": explained_variance
    }

Here, it was decided to use the Root Mean Squared Error, the Mean Absolute Error, the R-Squared Score and the Explained Variance Score. This set of metrics involves a good equilibrium between interpretability and robustness.

## Strategy 1 : Every combination's yield prediction

This strategy is straight-forward. A model takes as input a vector of descriptors containing information about the two reactives and the combination of reaction conditions. It would then return the expected yield for this particular combination. The model would then be used to predict the yield of every combination of ligand, base, and solvent with the input reactives. The results could then be sorted and the conditions selected. 

Here is a summary of the number of ligands, bases, and solvents:

| Element | Number |
|---|---|
| **Ligands** | 11 ligands + "no ligand" |
| **Bases** | 7 bases + "no base" | 
| **Solvents** | 6 solvents |

This amounts to 576 combinations when accounting for the possibility of no ligand or base. This method may seem slow and computationnaly expensive, but testing such a low amount of combinations is reasonable, especially if it allows the use of simpler strategies.


### Models to test

The following models will be tested to apply this strategy:

- Random Forest Regressor
- XGboost Regressor
- Deep Learning Model

All the models return a single value when given a vector as input.

### Model definition and training

**Random Forest**

This model is composed of **multiple trees** that are each trained on a different subset of the training dataset. When given an input, each tree processes the input vector and returns a single value. The output predicted value is **the average of all the proposed results**.

**XGBoost**

**Extreme gradient boosting** model is constructed by buildent decision trees one after the other. Each new tree attempts to correct the error of the previous one. It uses **gradient based optimization** to improve its performance. The final tree is used to predict the value of interest.

In [15]:
# Create models with grid search for hyperparameter tuning
# Temporarily defining x_train, y_train, x_test, y_test for avoidance of errors, replace with actual training and test data
rng = np.random.default_rng(42)

x_train = rng.normal(size=(100, 10))
coeff = np.arange(1, 11, dtype=float)
y_train = x_train.dot(coeff) + 2.0 + rng.normal(scale=0.5, size=100)

x_test = rng.normal(size=(20, 10))
y_test = x_test.dot(coeff) + 2.0 + rng.normal(scale=0.5, size=20)

# Grid search 
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20],
}

# Random Forest Regressor
grid_search_ranf = GridSearchCV(estimator=RandomForestRegressor(random_state=0), param_grid=param_grid, cv=5)
grid_search_ranf.fit(x_train, y_train)
print("Best parameters for Random Forest:", grid_search_ranf.best_params_)

# XGBoost Regressor
grid_search_xgb = GridSearchCV(estimator=XGBRegressor(random_state=0), param_grid=param_grid, cv=5)
grid_search_xgb.fit(x_train, y_train)
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)

Best parameters for Random Forest: {'max_depth': None, 'n_estimators': 100}
Best parameters for XGBoost: {'max_depth': None, 'n_estimators': 100}


In [16]:
# Test the models 
print(test_model(grid_search_ranf.best_estimator_, x_test, y_test))
print(test_model(grid_search_xgb.best_estimator_, x_test, y_test))

# Predict on the first 5 test vectors and display results
num_samples = 5
y_pred_ranf = grid_search_ranf.best_estimator_.predict(x_test[:num_samples])
y_pred_xgb = grid_search_xgb.best_estimator_.predict(x_test[:num_samples])

print("\nPredictions for first 5 test vectors:")
for i in range(num_samples):
    print(f"Tested vector {i+1}, Random Forest predicted yield: {y_pred_ranf[i]:.4f}, actual yield: {y_test[i]:.4f}")
    print(f"Tested vector {i+1}, XGBoost predicted yield: {y_pred_xgb[i]:.4f}, actual yield: {y_test[i]:.4f}")
    print("")

{'Model': 'RandomForestRegressor', 'RMSE': np.float64(12.761658671210064), 'MAE': 8.944060534388532, 'R2 Score': 0.5732359033966934, 'Explained Variance Score': 0.5934987273344929}
{'Model': 'XGBRegressor', 'RMSE': np.float64(10.835412070500096), 'MAE': 8.390034450689978, 'R2 Score': 0.6923446366796787, 'Explained Variance Score': 0.712488290181786}

Predictions for first 5 test vectors:
Tested vector 1, Random Forest predicted yield: 6.4487, actual yield: 11.2632
Tested vector 1, XGBoost predicted yield: 13.7312, actual yield: 11.2632

Tested vector 2, Random Forest predicted yield: -2.6296, actual yield: -3.2049
Tested vector 2, XGBoost predicted yield: 1.9771, actual yield: -3.2049

Tested vector 3, Random Forest predicted yield: -15.6607, actual yield: -14.5287
Tested vector 3, XGBoost predicted yield: -20.2680, actual yield: -14.5287

Tested vector 4, Random Forest predicted yield: 8.5938, actual yield: 11.3004
Tested vector 4, XGBoost predicted yield: 14.9669, actual yield: 11.30

**Deep Learning Model**

This model makes use of a neural networks consisting of **multiples interconnected node layers**. Each one applies a function to its inputs with different **weights**, and sends the output to the next layer. The last node will take the outputs of all the previous layer's nodes and calculate a single value which is the predicted reaction yield. The number of layers, layer size, dropout rate, learning rate, and other important hyperparameters will have to be adjusted to have a better performing model.

In [17]:
# Define a PyTorch Lightning model for regression (Copy paste from course for now)
class NeuralNetwork(pl.LightningModule):
    def __init__(self, input_sz, hidden_sz, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()  # saves input_size, hidden_size, lr

        self.layer_1 = torch.nn.Linear(input_sz, hidden_sz)
        self.layer_2 = torch.nn.Linear(hidden_sz, hidden_sz)
        self.layer_3 = torch.nn.Linear(hidden_sz, 1)

    def forward(self, x):
        # Returns the model predictions for input x / specified how data moves through the layers
        x = self.layer_1(x)
        x = F.relu(x)
        x = self.layer_2(x)
        x = F.relu(x)
        x = self.layer_3(x)
        return x # Shape: (batch_size, 1)

    def _shared_step(self, batch):
        x, y = batch
        preds = self(x)
        loss = F.mse_loss(preds, y)
        return loss

    def training_step(self, batch, batch_idx):
        # Here we define the train loop.
        loss = self._shared_step(batch)
        self.log("Train loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        # Define validation step. At the end of every epoch, this will be executed
        loss = self._shared_step(batch)
        self.log("Valid MSE", loss)

    def test_step(self, batch, batch_idx):
        # What to do in test
        loss = self._shared_step(batch)
        self.log("Test MSE", loss)

    def configure_optimizers(self):
        # Here we configure the optimization algorithm.
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr
        )
        return optimizer

In [18]:
# Define a custom dataset class for PyTorch
class ESOLDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Define a PyTorch Lightning DataModule for handling data loading
class NeuralNetworkDataModule(pl.LightningDataModule):
    def __init__(self, train_dataset, val_dataset, test_dataset, batch_size=256):
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [ ]:
# Initialize the model, data module, and trainer
input_size = x_train.shape[1]
hidden_size = 64
model = NeuralNetwork(input_size, hidden_size)
train_dataset = ESOLDataset(x_train, y_train)
val_dataset = ESOLDataset(x_test, y_test) 
test_dataset = ESOLDataset(x_test, y_test)
data_module = NeuralNetworkDataModule(train_dataset, val_dataset, test_dataset)
trainer = pl.Trainer(max_epochs=100)

# Train the model
trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name    | Type   | Params | Mode  | FLOPs
---------------------------------------------------
0 | layer_1 | Linear | 704    | train | 0    
1 | layer_2 | Linear | 4.2 K  | train | 0    
2 | layer_3 | Linear | 65     | train | 0    
---------------------------------------------------
4.9 K     Trainable params
0         Non-trainable params
4.9 K     Total params
0.020     Total estimated model params size (MB)
3         Modules in train mode
0         Modules in eval mode
0         Total Flops


c:\Users\corre\Documents\GitHub\.venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\corre\Documents\GitHub\.venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\corre\Documents\GitHub\.venv\lib\site-packages\pytorch_lightning\loops\fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 99: 100%|██████████| 1/1 [00:00<00:00, 41.66it/s, v_num=3] 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 1/1 [00:00<00:00, 33.89it/s, v_num=3]
{'Model': 'NeuralNetwork', 'RMSE': np.float64(8.033522727928512), 'MAE': 6.00945347523282, 'R2 Score': 0.8308836177761594, 'Explained Variance Score': 0.8336184554889497}


In [23]:
# Test the model
print(test_nn_model(model, x_test, y_test))

# Predict on the first 5 test vectors and display results
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    x_sample = torch.tensor(x_test[:5], dtype=torch.float32)
    y_pred_nn = model(x_sample).squeeze().numpy()

print("\nPredictions for first 5 test vectors (Neural Network):")
for i in range(num_samples):
    print(f"Tested vector {i+1}, Neural Network predicted yield: {y_pred_nn[i]:.4f}, actual yield: {y_test[i]:.4f}")
    print("")

{'Model': 'NeuralNetwork', 'RMSE': np.float64(8.033522727928512), 'MAE': 6.00945347523282, 'R2 Score': 0.8308836177761594, 'Explained Variance Score': 0.8336184554889497}

Predictions for first 5 test vectors (Neural Network):
Tested vector 1, Neural Network predicted yield: 5.1470, actual yield: 11.2632

Tested vector 2, Neural Network predicted yield: 1.2919, actual yield: -3.2049

Tested vector 3, Neural Network predicted yield: -9.8269, actual yield: -14.5287

Tested vector 4, Neural Network predicted yield: 8.4722, actual yield: 11.3004

Tested vector 5, Neural Network predicted yield: 15.4871, actual yield: 14.7010

